# Agriculture & Climate SLM — Student Starter

Welcome! You will fine-tune (or retrieve from) a small language model to generate
short text answers scored by **mean Levenshtein distance** (lower is better).

### What this notebook teaches

- How to load competition data on **Kaggle** (and locally for development).
- A **retrieval baseline** that runs on CPU.
- How to build **SFT training rows** from the public corpus.
- How to attach a pretrained model through **Kaggle Models** (no internet download at submit time).
- How to run a **quick supervised fine-tune** (full SFT) on a GPU.
- How to **LoRA fine-tune** as a lighter, parameter-efficient next step.
- How to validate and write `/kaggle/working/submission.csv`.
- How to submit from a committed **Kaggle Notebook** version.

> **Submission rule:** You must use a Kaggle Notebook. Run it from top to bottom, choose **Save Version → Save & Run All**, and submit `/kaggle/working/submission.csv` from the saved version's **Output** panel.

## 0. Kaggle notebook workflow

| Step | What to do |
|------|------------|
| 1 | Join the competition → **Code → New Notebook** (or open the official starter). |
| 2 | Confirm competition **data** is attached in the **Input** panel. |
| 3 | For fine-tuning: **Add Input → Models**. Enable **GPU** under **Settings → Accelerator**. |
| 4 | Run all cells, then **Save Version → Save & Run All**. |
| 5 | Submit `submission.csv` from the saved version's **Output** panel. |

Keep the **committed Kaggle notebook URL** if organizers ask. Do not submit a CSV created only on your laptop without a matching saved Kaggle run.

## 1. Load the competition files

On Kaggle, data is read-only under `/kaggle/input/`. Generated files go to `/kaggle/working/`.

- `train_qa.csv` — labelled training examples
- `test_questions.csv` — hidden test inputs
- `documents.csv` — reference corpus snippets

In [ ]:
from pathlib import Path
import pandas as pd

COMPETITION_SLUG = "agriculture-climate-slm"
ON_KAGGLE = Path("/kaggle/input").exists()

def find_data_dir(slug: str) -> Path:
    """Locate competition CSVs under /kaggle/input (layout varies slightly)."""
    if not ON_KAGGLE:
        return Path(".")
    root = Path("/kaggle/input")
    for pattern in (
        f"competitions/agriculture-climate-slm",
        f"competitions/{slug.replace('-', '_')}",
        slug,
    ):
        candidate = root / pattern
        if (candidate / "train_qa.csv").exists():
            return candidate
    for path in root.rglob("train_qa.csv"):
        return path.parent
    raise FileNotFoundError(
        "Could not find train_qa.csv. Join the competition and attach its dataset."
    )

DATA_DIR = find_data_dir(COMPETITION_SLUG)
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

docs = pd.read_csv(DATA_DIR / "documents.csv")
train = pd.read_csv(DATA_DIR / "train_qa.csv")
test = pd.read_csv(DATA_DIR / "test_questions.csv")
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print(len(docs), "docs ·", len(train), "train ·", len(test), "test")
display(train.head(2))


## 1b. Retrieval baseline (CPU, no fine-tuning)

Copy the nearest train answer by topic-filtered TF-IDF similarity.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

rows = []
for _, t in test.iterrows():
    sub = train[train["topic"] == t["topic"]]
    if sub.empty:
        sub = train
    vec = TfidfVectorizer(ngram_range=(1, 2))
    x_train = vec.fit_transform(sub["question"])
    x_test = vec.transform([t["question"]])
    sims = cosine_similarity(x_test, x_train)
    ans = sub.iloc[sims.argmax()]["reference_answer"]
    rows.append({"QuestionId": t["QuestionId"], "Answer": ans})

baseline_submission = pd.DataFrame(rows)
baseline_submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print("Wrote", OUTPUT_DIR / "submission.csv")
display(baseline_submission.head())


## 2. Validate and save the submission file

Kaggle expects exactly `QuestionId,Answer` — one row per test item, same order as `test_*`.

Write only to **`/kaggle/working/`** on Kaggle (`/kaggle/input` is read-only).

In [ ]:
assert list(baseline_submission.columns) == ["QuestionId", "Answer"]
assert len(baseline_submission) == len(test)
assert baseline_submission["QuestionId"].tolist() == test["QuestionId"].tolist()
assert baseline_submission["QuestionId"].is_unique
assert baseline_submission["Answer"].notna().all()

output_path = OUTPUT_DIR / "submission.csv"
baseline_submission.to_csv(output_path, index=False)
print("Validated and saved", len(baseline_submission), "rows →", output_path)


## 2. Build a RAG examples

### a. Building the index in my docs

In [ ]:
## Transforming my documents into vectors

vectorizer = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    stop_words="english"
)

document_vectors = vectorizer.fit_transform(docs["text"])

print("Document matrix:", document_vectors.shape)

### b. creating the reasearch function

In [ ]:
import numpy as np

def retrieve_documents(query, top_k=3):
    """
    Retrieve the top-k most relevant documents for a query.
    """

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        document_vectors
    ).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = docs.iloc[top_indices].copy()
    results["retrieval_score"] = scores[top_indices]

    return results

In [ ]:
## Let's test our retreavial build

query = "What are the effects of drought on crop production?"

results = retrieve_documents(query, top_k=3)

for _, row in results.iterrows():
    print("=" * 80)
    print("Score:", row["retrieval_score"])
    print(row["text"][:1000])

### c. Building our RAG prompt

In [ ]:
def build_prompt_for_test(row, top_k=3):
    question = row["question"]

    retrieved = retrieve_documents(
        question,
        top_k=top_k
    )

    context_parts = []

    for i, (_, doc) in enumerate(retrieved.iterrows(), start=1):
        context_parts.append(
            f"[Document {i}]\n{doc['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are answering a question using the provided documents.

Use the documents to answer the question.
Do not invent information that is not supported by the documents.

Documents:
{context}

Question:
{question}

Answer:"""

    return prompt

### d. let's create a generate rag output

## 3. Build SFT examples

Include **crop** and **agro_zone** metadata in every prompt. Optional: TF-IDF retrieval over `documents.csv`.

In [ ]:
def build_prompt(row, docs_df, context_chars=400):
    doc = docs_df.loc[docs_df.document_id == row.document_id, "text"].iloc[0]
    return (
        f"Crop: {row.crop} | Zone: {row.agro_zone} | Topic: {row.topic}\n"
        f"Question: {row.question}\n"
        f"Context: {doc[:context_chars]}\n"
        f"Answer:"
    )

# def build_prompt_for_test(t):
#     sub = train[train["topic"] == t.topic]
#     row = sub.iloc[0] if len(sub) else train.iloc[0]
#     doc = docs.loc[docs.document_id == row.document_id, "text"].iloc[0]
#     return (
#         f"Crop: {t.crop} | Zone: {t.agro_zone} | Topic: {t.topic}\n"
#         f"Question: {t.question}\n"
#         f"Context: {doc[:400]}\n"
#         f"Answer:"
#     )

sft = [{"text": build_prompt(r, docs) + " " + r.reference_answer} for _, r in train.iterrows()]
print(len(sft), "SFT rows")
print(sft[0])

## 4. Attach a pretrained model from Kaggle Models

Avoid downloading weights from the internet during **Save & Run All**:

1. Open the notebook **Input** panel (right side) → **Add Input** → **Models**.
2. Search for a small instruct model, e.g. **`google/gemma-2-2b-it`** or **`microsoft/Phi-3-mini-4k-instruct`**.
3. Attach a **Hugging Face Transformers / PyTorch** variation with `config.json`, tokenizer files, and
   `model.safetensors` or `pytorch_model.bin`.
4. **Settings → Accelerator → GPU** (T4 is enough for LoRA on 1–3B models).
5. Run the next cell. It sets `MODEL_PATH` under `/kaggle/input` and loads with `local_files_only=True`.

If multiple models are attached, set `MODEL_PATH` manually to the folder you want.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

model_candidates = ["/kaggle/input/models/google/gemma/transformers/2b-it/3"]
for config_file in Path("/kaggle/input").rglob("config.json") if ON_KAGGLE else []:
    folder = config_file.parent
    has_tok = (folder / "tokenizer.json").exists() or (folder / "tokenizer_config.json").exists()
    has_weights = (folder / "model.safetensors").exists() or (folder / "pytorch_model.bin").exists()
    if has_tok and has_weights:
        model_candidates.append(folder)

if ON_KAGGLE and model_candidates:
    MODEL_PATH = str(sorted(model_candidates, key=lambda p: len(str(p)))[0])
    print("Using attached Kaggle Model:", MODEL_PATH)
else:
    MODEL_PATH = "google/gemma-2-2b-it"  # local dev only — needs HF download
    print("Local/dev mode — will download from Hugging Face:", MODEL_PATH)


## 4b. Quick supervised fine-tune (full SFT)

**Supervised fine-tuning (SFT)** updates *all* model weights on your prompt→answer pairs.
This is the classic fine-tuning step before parameter-efficient methods like LoRA.

Defaults: **1 epoch**, `lr=2e-5`, max length 512. Set `USE_SFT = True` after attaching
a model and enabling GPU. For a faster demo, set `SFT_TRAIN_MAX_ROWS = 64` (uses a subset only).

> Full SFT uses more GPU memory than LoRA. If you hit OOM, lower batch size or skip to §4c.

In [ ]:
%%capture
!pip install trl
!pip install --upgrade peft transformers
!pip install --upgrade torchao


In [ ]:
USE_SFT = False  # set True after attaching a model + enabling GPU
SFT_TRAIN_MAX_ROWS = None  # e.g. 64 for a quick demo subset
SFT_EPOCHS = 1
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 96
LOCAL_ONLY = ON_KAGGLE

model = None
tok = None

def load_model_and_tokenizer():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=LOCAL_ONLY)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=LOCAL_ONLY,
    )
    return tokenizer, base_model

# def generate_output(active_model, active_tok, row):
#     import torch

#     prompt = build_prompt_for_test(row)
#     inputs = active_tok(prompt, return_tensors="pt").to(active_model.device)
#     with torch.no_grad():
#         out = active_model.generate(
#             **inputs,
#             max_new_tokens=MAX_NEW_TOKENS,
#             do_sample=False,
#             pad_token_id=active_tok.eos_token_id,
#         )
#     decoded = active_tok.decode(out[0], skip_special_tokens=True)
#     return decoded.split("Answer:")[-1].strip()


def generate_output(active_model, active_tok, row, top_k=3):

    prompt = build_prompt_for_test(row, top_k=top_k)

    inputs = active_tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    ).to(active_model.device)

    with torch.no_grad():
        out = active_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=active_tok.eos_token_id,
        )

    decoded = active_tok.decode(
        out[0],
        skip_special_tokens=True
    )

    return decoded.split("Answer:")[-1].strip()

if USE_SFT:
    import torch
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer

    tok, model = load_model_and_tokenizer()
    train_rows = sft[:SFT_TRAIN_MAX_ROWS] if SFT_TRAIN_MAX_ROWS else sft
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    # Si le BF16 n'est pas supporté, on se rabat sur le FP16
    use_fp16 = torch.cuda.is_available() and not use_bf16
    print(f"Full SFT on {{len(train_rows)}} rows for {{SFT_EPOCHS}} epoch(s)")

    SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR / "sft-checkpoints"),
            num_train_epochs=SFT_EPOCHS,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-5,
            logging_steps=5,
            save_strategy="no",
            max_length=MAX_SEQ_LENGTH,
            dataset_text_field="text",
            fp16=use_fp16,
            bf16=use_bf16,
            loss_type="nll"
        ),
        train_dataset=Dataset.from_dict({"text": [row["text"] for row in train_rows]}),
        processing_class=tok,
        
    ).train()

    sft_submission = pd.DataFrame([
        {"QuestionId": t["QuestionId"], "Answer": generate_output(model, tok, t)}
        for _, t in test.iterrows()
    ])
    sft_submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("Wrote full SFT submission →", OUTPUT_DIR / "submission.csv")
else:
    print("Set USE_SFT=True for full supervised fine-tuning (§4b), or USE_LORA=True for LoRA (§4c).")


## 4c. Quick LoRA fine-tune

**LoRA** trains only low-rank adapter weights (~1–2% of parameters) — faster and lighter than full SFT.
If you ran §4b, LoRA continues from that fine-tuned model; otherwise it starts from the base checkpoint.

Defaults: `r=8`, 3 epochs, `lr=2e-4`. Set `USE_LORA = True` after attaching a model and enabling GPU.

In [ ]:
USE_LORA = True  # set True after attaching a model + enabling GPU
LOCAL_ONLY = ON_KAGGLE

if USE_LORA:
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from trl import SFTConfig, SFTTrainer

    if model is None or tok is None:
        tok, model = load_model_and_tokenizer()

    model = get_peft_model(
        model,
        LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            task_type=TaskType.CAUSAL_LM,
        ),
    )
    model.print_trainable_parameters()

    SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR / "lora-checkpoints"),
            num_train_epochs=17,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            logging_steps=5,
            save_strategy="no",
            max_length=MAX_SEQ_LENGTH,
            dataset_text_field="text",
            loss_type="nll",
            fp16=torch.cuda.is_available(),
        ),
        train_dataset=Dataset.from_dict({"text": [row["text"] for row in sft]}),
        processing_class=tok,
    ).train()

    lora_submission = pd.DataFrame([
        {"QuestionId": t["QuestionId"], "Answer": generate_output(model, tok, t)}
        for _, t in test.iterrows()
    ])
    lora_submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("Wrote LoRA submission →", OUTPUT_DIR / "submission.csv")
else:
    print("Set USE_LORA=True to run LoRA fine-tuning on top of §4b (or from the base model).")


## 5. Submit from your committed Kaggle Notebook

1. Confirm the latest cell wrote **`submission.csv`** to `/kaggle/working/`.
2. Check that the file contains exactly **`QuestionId,Answer`**, one row per test item, and non-empty text.
3. Click **Save Version**, choose **Save & Run All**, and wait for the committed run to finish successfully. An interactive draft is not a valid final notebook.
4. Open the saved version's **Output** panel, confirm `submission.csv` is present, and select **Submit to Competition**. If that button is unavailable, download the output CSV and upload it on the competition submission page, then provide the committed notebook link as required by the rules.
5. Keep the submitted notebook private during the competition unless organizers explicitly request publication.

The notebook must be reproducible from top to bottom and must not read the private solution file or hard-code hidden test labels.

| Problem | What to check |
|---------|----------------|
| `FileNotFoundError` for CSVs | Join competition; data must appear in **Input** panel. |
| No attached model found | **Add Input → Models** before §4b/§4c. |
| CUDA OOM on full SFT | Lower batch size, set `SFT_TRAIN_MAX_ROWS`, or skip to LoRA (§4c). |
| Kaggle rejects CSV | Check row count, column names, unique IDs, non-empty text. |

**Disclaimer:** Synthetic extension text — not certified agronomic advice.